# 02b — Train SAE on cached activations
### Driver is thin, heavy lifting is done in the mid.sae.train_sae
Owner: David Teklea

In [ ]:
import sys
import torch
sys.path.insert(0, "../src")

from mid.sae.activations import read_activations
from mid.sae.load import load_sae
from mid.sae.train_sae import train_sae
from mid.config import load_configs, load_sae_config


In [ ]:
#Model type: String: [decoder, encoder, or encoderdecoder]  Default: decoder
model_type = "decoder"
#Model size: String: [small, medium, large]  Default: small
model_size = "small"

#Use split: Bool: [True, False]  Default: True
#Whether to split data into train and validation for the SAE or to use all data.
use_split = True # True -> train.pt; False -> all.pt

model_config_path = f"../configs/model_{model_type}_{model_size}.yaml"
sae_config_path = f"../configs/sae_{model_type}_{model_size}.yaml"
checkpoint_path = f"../checkpoints/{model_type}_{model_size}_best_model.pt"
activations_pt_path = (
    f"../data/sae_data/{model_type}_{model_size}/train.pt" if use_split
    else f"../data/sae_data/{model_type}_{model_size}/all.pt"
)

model_cfg, _ = load_configs(model_config_path)
sae_cfg = load_sae_config(sae_config_path)
print(model_cfg)
print(sae_cfg)


In [ ]:
#Collect input data

#Specify the layer here, or you can make this a for loop across all layers
layer = 0

if use_split:
    #Training
    get_path = f"../data/sae_data/{model_type}_{model_size}/train.pt"
    train_activations, train_metadata, train_num_tokens, train_num_activations = read_activations(get_path, layer)
    #Validation
    get_path = f"../data/sae_data/{model_type}_{model_size}/val.pt"
    train_activations, train_metadata, train_num_tokens, train_num_activations = read_activations(get_path, layer)

else:
    #All, no validation, saved as training to avoid having to have another variable name.
    get_path = f"../data/sae_data/{model_type}_{model_size}/all.pt"
    train_activations, train_metadata, train_num_tokens, train_num_activations = read_activations(get_path, layer)

In [ ]:
# Train a single SAE. Output goes to `../data/sae_out/{model_type}_{model_size}/L{layer}_{hook_type}/`.

In [ ]:
out_dir = (
    f"../data/sae_out/{model_type}_{model_size}/L{sae_cfg.layer}_{sae_cfg.hook_type}"
)

sae = train_sae(
    sae_cfg=sae_cfg,
    model_cfg=model_cfg,
    checkpoint_path=checkpoint_path,
    activations_pt_path=activations_pt_path,
    out_dir=out_dir,
)

In [ ]:
# Smoke test: reload the SAE from disk and check reconstruction + sparsity on
# a slice of the cached train activations. Feature density (L0) should be a small fraction
# of d_sae; variance-explained should be close to 1.

device = "cuda" if torch.cuda.is_available() else "cpu"
sae = load_sae(out_dir, device=device)

activations, _, _, _ = read_activations(activations_pt_path, layer_num=sae_cfg.layer)
x = activations[:4096].to(device, dtype=torch.float32)

with torch.no_grad():
    feats = sae.encode(x)
    x_hat = sae.decode(feats)

l0 = (feats > 0).float().sum(dim=-1).mean().item()
mse = torch.nn.functional.mse_loss(x_hat, x).item()
var_explained = 1 - ((x - x_hat).var() / x.var()).item()
print(f"L0 (mean active features per token): {l0:.1f} / {sae_cfg.d_sae}")
print(f"Reconstruction MSE: {mse:.4f}")
print(f"Variance explained: {var_explained:.3f}")